# FloodShift 08 — Grounded Multimodal Instruction Dataset

This notebook converts the verified **FloodShift-07 deterministic evidence packets** into a grounded instruction dataset for the later variational VLM.

It performs **no segmentation training** and **no language-model training**.

The central rule is:

\[
\boxed{\text{Every factual answer must be traceable to a deterministic FloodShift-07 fact.}}
\]

No answer is generated by an external LLM.

## Source artifact

Expected accepted archive:

```text
FloodShift_07_ARCHIVE.zip
SHA256:
1a8a5c35f2afedeb92aa58d7c161bca22bbaee762d6533709a515f3798a8344b
```

The archive contains:

- 34 verified scene evidence packets,
- 34 grounding-safe evidence panels,
- fact schema,
- scene summary,
- protocol record.

## Why this stage exists

The later VLM must learn four distinct behaviors:

1. **Grounded description**
   - predicted temporary-flood extent,
   - permanent water,
   - spatial geometry.

2. **Reliability-aware explanation**
   - predictive entropy,
   - review fractions,
   - \(P(\mathrm{clear})\).

3. **Counterfactual sensor attribution**
   - SAR dependence,
   - optical dependence,
   - prediction fragility,
   - without claiming physical causality.

4. **Abstention / refusal**
   - do not invent flood causes,
   - do not invent population or damage estimates,
   - do not forecast future flooding from a static evidence packet,
   - do not call model output ground truth.

## Scene-disjoint split

Examples are split **by scene**, never by question.

This prevents near-duplicate questions from the same image appearing in both instruction train and instruction validation.

The frozen geographic segmentation test is still untouched.

## Dataset task families

Each scene receives exactly 10 deterministic instruction examples:

1. extent
2. uncertainty
3. sensor attribution
4. temporary-flood sensor attribution
5. spatial geometry
6. integrated scientific summary
7. review / abstention policy
8. physical-cause refusal
9. human-impact refusal
10. future-forecast refusal

With 34 scenes this produces:

\[
34\times10 = 340
\]

grounded multimodal examples.


In [ ]:
# ============================================================
# CELL: P16-C01 — ENVIRONMENT
# ============================================================
import os
import sys
import json
import math
import hashlib
import zipfile
import shutil
import random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/FloodShift_08_Grounded_Instructions")
EXTRACT_ROOT = OUTPUT_ROOT / "_source07"
DATASET_ROOT = OUTPUT_ROOT / "dataset"
IMAGE_ROOT = DATASET_ROOT / "images"

for p in [
    OUTPUT_ROOT,
    EXTRACT_ROOT,
    DATASET_ROOT,
    IMAGE_ROOT,
]:
    p.mkdir(parents=True, exist_ok=True)

EXPECTED_07_ARCHIVE_SHA256 = (
    "1a8a5c35f2afedeb92aa58d7c161bca22bbaee762d6533709a515f3798a8344b"
)

print("=" * 112)
print("P16-C01 — GROUNDED INSTRUCTION DATASET ENVIRONMENT")
print("=" * 112)
print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("Output root :", OUTPUT_ROOT)


In [ ]:
# ============================================================
# CELL: P16-C02 — LOCATE + VERIFY + EXTRACT NOTEBOOK-07 ARCHIVE
# ============================================================
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def locate(filename):
    matches = list(INPUT_ROOT.rglob(filename))
    if not matches:
        raise FileNotFoundError(
            f"Could not find '{filename}' under /kaggle/input."
        )
    return sorted(
        matches,
        key=lambda p: (len(str(p)), str(p)),
    )[0]

ARCHIVE_07 = locate("FloodShift_07_ARCHIVE.zip")
ARCHIVE_07_SHA = sha256_file(ARCHIVE_07)

assert ARCHIVE_07_SHA == EXPECTED_07_ARCHIVE_SHA256, (
    "Notebook-07 archive SHA mismatch.\n"
    f"Expected: {EXPECTED_07_ARCHIVE_SHA256}\n"
    f"Found   : {ARCHIVE_07_SHA}"
)

with zipfile.ZipFile(ARCHIVE_07, "r") as zf:
    zf.extractall(EXTRACT_ROOT)

PACKETS_PATH = (
    EXTRACT_ROOT
    / "packets"
    / "FloodShift_07_scene_evidence_packets.jsonl"
)

SUMMARY_PATH = (
    EXTRACT_ROOT
    / "metrics"
    / "FloodShift_07_scene_summary.csv"
)

SCHEMA_PATH = (
    EXTRACT_ROOT
    / "FloodShift_07_fact_schema.json"
)

PROTOCOL_PATH = (
    EXTRACT_ROOT
    / "FloodShift_07_protocol_record.json"
)

PANEL_ROOT_07 = EXTRACT_ROOT / "panels"

for p in [
    PACKETS_PATH,
    SUMMARY_PATH,
    SCHEMA_PATH,
    PROTOCOL_PATH,
]:
    assert p.exists(), f"Missing expected Notebook-07 artifact: {p}"

with open(PROTOCOL_PATH, "r") as f:
    protocol07 = json.load(f)

with open(SCHEMA_PATH, "r") as f:
    fact_schema07 = json.load(f)

assert protocol07["source_segmenter"] == "FloodShift-04H-CAL"
assert protocol07["05b_status"] == "NOT_PROMOTED_NOT_USED"
assert protocol07["packet_count"] == 34
assert protocol07["packet_verifier_failures"] == 0
assert protocol07["ground_truth_in_packet"] is False
assert protocol07["test_status"] == "LOCKED_NOT_EVALUATED"

print("=" * 112)
print("P16-C02 — NOTEBOOK-07 SOURCE VERIFIED")
print("=" * 112)
print("Archive       :", ARCHIVE_07)
print("SHA256        :", ARCHIVE_07_SHA)
print("Packet count  :", protocol07["packet_count"])
print("Segmenter     :", protocol07["source_segmenter"])
print("05B status    :", protocol07["05b_status"])
print("Ground truth  :", protocol07["ground_truth_in_packet"])
print("Frozen test   :", protocol07["test_status"])
print("P16-C02 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P16-C03 — LOAD + REVERIFY ALL SCENE PACKETS / PANELS
# ============================================================
packets = []

with open(PACKETS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            packets.append(json.loads(line))

assert len(packets) == 34

scene_ids = [p["scene_id"] for p in packets]
assert len(scene_ids) == len(set(scene_ids))

def forbidden_packet_content(packet):
    text = json.dumps(packet).lower()
    forbidden = [
        '"ground_truth"',
        '"target"',
        '"label_mask"',
        '"true_class"',
    ]
    return [tok for tok in forbidden if tok in text]

panel_records = []

for packet in packets:
    assert packet["schema_version"] == "FloodShift-GeoEvidence-v1"
    assert packet["provenance"]["segmenter"] == "FloodShift-04H-CAL"
    assert packet["provenance"]["verifier_status"] == "PASS"
    assert packet["provenance"]["test_status"] == "LOCKED_NOT_EVALUATED"
    assert packet["visual_evidence"]["contains_ground_truth"] is False

    forbidden = forbidden_packet_content(packet)
    assert not forbidden, (
        f"Forbidden content in {packet['scene_id']}: {forbidden}"
    )

    panel = PANEL_ROOT_07 / f"{packet['scene_id']}_evidence.png"
    assert panel.exists(), f"Missing evidence panel: {panel}"

    with Image.open(panel) as im:
        w, h = im.size
        assert w > 0 and h > 0

    panel_records.append({
        "scene_id": packet["scene_id"],
        "panel_path": str(panel),
        "width": w,
        "height": h,
    })

print("=" * 112)
print("P16-C03 — PACKET / PANEL AUDIT")
print("=" * 112)
print("Verified packets :", len(packets))
print("Verified panels  :", len(panel_records))
print("Ground-truth leak: 0")
print("P16-C03 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P16-C04 — SCENE-DISJOINT INSTRUCTION TRAIN / VAL SPLIT
# ============================================================
def split_hash(scene_id):
    token = ("FloodShift08|" + str(scene_id)).encode("utf-8")
    return hashlib.sha256(token).hexdigest()

split_df = pd.DataFrame({
    "scene_id": scene_ids,
})

split_df["hash"] = split_df["scene_id"].map(split_hash)
split_df = split_df.sort_values("hash").reset_index(drop=True)

# 27 / 7 scene split.
TRAIN_SCENES = set(
    split_df.iloc[:27]["scene_id"].tolist()
)

VAL_SCENES = set(
    split_df.iloc[27:]["scene_id"].tolist()
)

assert len(TRAIN_SCENES) == 27
assert len(VAL_SCENES) == 7
assert TRAIN_SCENES.isdisjoint(VAL_SCENES)
assert TRAIN_SCENES | VAL_SCENES == set(scene_ids)

scene_split = {
    sid: (
        "train"
        if sid in TRAIN_SCENES
        else "validation"
    )
    for sid in scene_ids
}

print("=" * 112)
print("P16-C04 — SCENE SPLIT")
print("=" * 112)
print("Train scenes      :", len(TRAIN_SCENES))
print("Validation scenes :", len(VAL_SCENES))
print("Scene overlap     :", len(TRAIN_SCENES & VAL_SCENES))
print("Frozen geo test   : untouched")
print("P16-C04 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P16-C05 — FIXED TEXT / ABSTENTION POLICY
# ============================================================
# These are response-policy thresholds for instruction generation.
# They are NOT segmentation metrics and are not tuned on labels.
SCENE_REVIEW_PIXEL_FRACTION = 0.20
SCENE_FRAGILITY_TILE_FRACTION = 0.50

def pct(x, decimals=2):
    if x is None:
        return "unavailable"
    return f"{100.0 * float(x):.{decimals}f}%"

def dec(x, decimals=3):
    if x is None:
        return "unavailable"
    return f"{float(x):.{decimals}f}"

def area(x):
    if x is None:
        return "unavailable"
    return f"{float(x):.2f} km²"

def integer(x):
    return str(int(x))

def scene_requires_review(packet):
    review10 = float(
        packet["uncertainty"]["review10_fraction"]
    )
    fragile = float(
        packet["counterfactual_attribution"][
            "high_fragility_tile_fraction"
        ]
    )

    return (
        review10 >= SCENE_REVIEW_PIXEL_FRACTION
        or fragile >= SCENE_FRAGILITY_TILE_FRACTION
    )

def example_id(scene_id, task):
    return f"{scene_id}__{task}"

print("=" * 112)
print("P16-C05 — RESPONSE POLICY")
print("=" * 112)
print(
    "Scene review rule:",
    f"review10 >= {SCENE_REVIEW_PIXEL_FRACTION:.2f}",
    "OR",
    f"high-fragility-tile fraction >= {SCENE_FRAGILITY_TILE_FRACTION:.2f}",
)
print("No label-derived thresholding is used.")
print("P16-C05 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P16-C06 — DETERMINISTIC TASK GENERATOR
# ============================================================
TASKS = [
    "extent",
    "uncertainty",
    "sensor_attribution",
    "temporary_sensor_attribution",
    "spatial_geometry",
    "integrated_summary",
    "review_policy",
    "cause_refusal",
    "impact_refusal",
    "forecast_refusal",
]

def make_example(packet, task):
    sid = packet["scene_id"]
    split = scene_split[sid]

    seg = packet["segmentation"]
    unc = packet["uncertainty"]
    sq = packet["sensor_quality"]
    att = packet["counterfactual_attribution"]
    geo = packet["spatial_geometry"]
    prov = packet["provenance"]

    tmp_frac = seg["temporary_flood"]["fraction_of_coverage"]
    tmp_area = seg["temporary_flood"]["area_km2"]
    perm_frac = seg["permanent_water"]["fraction_of_coverage"]

    mean_ent = unc["mean_entropy"]
    review10 = unc["review10_fraction"]
    review20 = unc["review20_fraction"]

    mean_pclear = sq["mean_pclear"]
    low_pclear = sq["low_pclear_fraction"]

    sar_share = att["sar_share"]
    opt_share = att["optical_share"]
    dominance = att["dominance"]
    flip_sar = att["flip_rate_no_sar"]
    flip_opt = att["flip_rate_no_optical"]

    tmp_js_sar = att["temporary_flood_js_sar"]
    tmp_js_opt = att["temporary_flood_js_optical"]

    high_frag = att["high_fragility_tile_fraction"]
    mean_frag = att["mean_fragility"]

    facts = []
    mode = "answer"

    if task == "extent":
        instruction = (
            "Using only the verified FloodShift evidence, summarize the "
            "predicted temporary-flood and permanent-water extent in this scene."
        )

        response = (
            f"FloodShift predicts temporary flooding over {pct(tmp_frac)} "
            f"of the sensor-covered scene"
        )

        if tmp_area is not None:
            response += f", corresponding to {area(tmp_area)}"

        response += (
            f". Permanent water accounts for {pct(perm_frac)} of the "
            "sensor-covered scene. These values are model-derived predictions, "
            "not ground truth."
        )

        facts = [
            "segmentation.temporary_flood.fraction_of_coverage",
            "segmentation.temporary_flood.area_km2",
            "segmentation.permanent_water.fraction_of_coverage",
        ]

    elif task == "uncertainty":
        instruction = (
            "How uncertain is the FloodShift prediction for this scene? "
            "Use only the calibrated uncertainty facts."
        )

        response = (
            f"The mean normalized predictive entropy is {dec(mean_ent)}. "
            f"{pct(review10)} of sensor-covered pixels exceed the frozen "
            f"review-10 threshold, and {pct(review20)} exceed the frozen "
            f"review-20 threshold. Mean optical reliability P(clear) is "
            f"{dec(mean_pclear)}, with {pct(low_pclear)} of covered pixels "
            f"below P(clear)=0.50."
        )

        facts = [
            "uncertainty.mean_entropy",
            "uncertainty.review10_fraction",
            "uncertainty.review20_fraction",
            "sensor_quality.mean_pclear",
            "sensor_quality.low_pclear_fraction",
        ]

    elif task == "sensor_attribution":
        instruction = (
            "Which sensor does the current FloodShift prediction depend on "
            "more strongly, and what does that attribution mean?"
        )

        response = (
            f"The counterfactual attribution is {dominance}. "
            f"Optical contributes {pct(opt_share)} of the normalized "
            f"SAR-versus-optical attribution share and SAR contributes "
            f"{pct(sar_share)}. Removing optical changes the predicted class "
            f"for {pct(flip_opt)} of covered pixels, while removing SAR changes "
            f"{pct(flip_sar)}. These scores measure model dependence under "
            f"counterfactual sensor removal; they do not establish physical causality."
        )

        facts = [
            "counterfactual_attribution.dominance",
            "counterfactual_attribution.optical_share",
            "counterfactual_attribution.sar_share",
            "counterfactual_attribution.flip_rate_no_optical",
            "counterfactual_attribution.flip_rate_no_sar",
        ]

    elif task == "temporary_sensor_attribution":
        instruction = (
            "For pixels predicted as temporary flood, compare FloodShift's "
            "counterfactual sensitivity to SAR removal and optical removal."
        )

        if tmp_js_sar is None or tmp_js_opt is None:
            response = (
                "This evidence packet does not contain a valid temporary-flood "
                "counterfactual attribution value for this scene, so a "
                "temporary-flood-specific sensor comparison is not supported."
            )
            mode = "abstain"
        else:
            dominant_tmp = (
                "optical"
                if float(tmp_js_opt) > float(tmp_js_sar)
                else "SAR"
            )

            response = (
                f"Within pixels predicted as temporary flood, normalized "
                f"counterfactual JS sensitivity is {dec(tmp_js_opt)} for optical "
                f"removal and {dec(tmp_js_sar)} for SAR removal. The temporary-"
                f"flood prediction is therefore more sensitive to {dominant_tmp} "
                f"removal in this scene. This is model-dependence evidence, "
                f"not physical causality."
            )

        facts = [
            "counterfactual_attribution.temporary_flood_js_optical",
            "counterfactual_attribution.temporary_flood_js_sar",
        ]

    elif task == "spatial_geometry":
        instruction = (
            "Describe the deterministic spatial geometry of the predicted "
            "temporary-flood mask without interpreting connected components "
            "as separate flood events."
        )

        count = geo[
            "temporary_flood_connected_components_8conn"
        ]

        largest_px = geo[
            "largest_temporary_component_pixels"
        ]

        largest_area = geo[
            "largest_temporary_component_area_km2"
        ]

        response = (
            f"The predicted temporary-flood mask contains {integer(count)} "
            f"pixel-level 8-connected components. The largest component contains "
            f"{integer(largest_px)} pixels"
        )

        if largest_area is not None:
            response += f" and covers {area(largest_area)}"

        response += (
            ". These connected components are image-mask structures and must "
            "not be interpreted as distinct real-world flood events."
        )

        facts = [
            "spatial_geometry.temporary_flood_connected_components_8conn",
            "spatial_geometry.largest_temporary_component_pixels",
            "spatial_geometry.largest_temporary_component_area_km2",
        ]

    elif task == "integrated_summary":
        instruction = (
            "Give a concise scientific summary of the verified FloodShift "
            "evidence for this scene. Include extent, uncertainty, optical "
            "reliability, and sensor dependence."
        )

        response = (
            f"FloodShift predicts temporary flooding over {pct(tmp_frac)} "
            f"of the sensor-covered scene"
        )

        if tmp_area is not None:
            response += f" ({area(tmp_area)})"

        response += (
            f". Mean normalized predictive entropy is {dec(mean_ent)}, and "
            f"{pct(review10)} of covered pixels exceed the frozen review-10 "
            f"threshold. Mean P(clear) is {dec(mean_pclear)}. Counterfactual "
            f"sensor attribution is {dominance}, with optical share "
            f"{pct(opt_share)} and SAR share {pct(sar_share)}. The attribution "
            f"describes model dependence, not physical causality."
        )

        facts = [
            "segmentation.temporary_flood.fraction_of_coverage",
            "segmentation.temporary_flood.area_km2",
            "uncertainty.mean_entropy",
            "uncertainty.review10_fraction",
            "sensor_quality.mean_pclear",
            "counterfactual_attribution.dominance",
            "counterfactual_attribution.optical_share",
            "counterfactual_attribution.sar_share",
        ]

    elif task == "review_policy":
        instruction = (
            "Should this scene be flagged for additional human review under "
            "the fixed FloodShift instruction-dataset review policy?"
        )

        needs_review = scene_requires_review(packet)

        if needs_review:
            response = (
                f"Yes. Under the fixed scene-level review policy, this scene is "
                f"flagged because {pct(review10)} of covered pixels exceed the "
                f"frozen review-10 entropy threshold and {pct(high_frag)} of its "
                f"tiles exceed the frozen counterfactual-fragility threshold. "
                f"This flag requests review; it does not declare the prediction incorrect."
            )
            mode = "abstain"
        else:
            response = (
                f"No additional scene-level flag is triggered by the fixed policy: "
                f"{pct(review10)} of covered pixels exceed the review-10 entropy "
                f"threshold and {pct(high_frag)} of tiles exceed the frozen "
                f"counterfactual-fragility threshold. This does not make the "
                f"prediction ground truth."
            )

        facts = [
            "uncertainty.review10_fraction",
            "counterfactual_attribution.high_fragility_tile_fraction",
        ]

    elif task == "cause_refusal":
        instruction = (
            "What caused the flooding in this scene? Identify the physical cause."
        )

        response = (
            "The verified FloodShift evidence does not establish the physical "
            "cause of flooding. Counterfactual SAR/optical attribution measures "
            "the model's dependence on sensor inputs, not environmental causality. "
            "A physical-cause claim would require external hydrologic, meteorological, "
            "or event data that are not present in this packet."
        )

        facts = []
        mode = "refuse"

    elif task == "impact_refusal":
        instruction = (
            "How many people or buildings were affected by the predicted flooding?"
        )

        response = (
            "The FloodShift evidence packet does not contain population, building, "
            "damage, or exposure data. It supports predicted flood extent and "
            "model-reliability facts only, so the number of affected people or "
            "buildings cannot be determined from this evidence."
        )

        facts = []
        mode = "refuse"

    elif task == "forecast_refusal":
        instruction = (
            "Will flooding in this scene become worse tomorrow?"
        )

        response = (
            "This static FloodShift evidence packet does not provide a future "
            "forecast. It describes the current model prediction, uncertainty, "
            "sensor reliability, and counterfactual sensor dependence. Predicting "
            "tomorrow's flood evolution would require temporal or forecast inputs "
            "that are not contained in this evidence."
        )

        facts = []
        mode = "refuse"

    else:
        raise ValueError(task)

    return {
        "id": example_id(sid, task),
        "scene_id": sid,
        "split": split,
        "task": task,
        "image": f"images/{sid}_evidence.png",
        "instruction": instruction,
        "response": response,
        "response_mode": mode,
        "grounded_fact_paths": facts,
        "source_packet_sha256": prov["packet_sha256"],
        "source_segmenter": prov["segmenter"],
        "source_test_status": prov["test_status"],
    }

print("P16-C06 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P16-C07 — GENERATE ALL 340 EXAMPLES
# ============================================================
examples = []

for packet in packets:
    for task in TASKS:
        examples.append(
            make_example(
                packet,
                task,
            )
        )

assert len(examples) == 340
assert len({x["id"] for x in examples}) == 340

train_examples = [
    x for x in examples
    if x["split"] == "train"
]

val_examples = [
    x for x in examples
    if x["split"] == "validation"
]

assert len(train_examples) == 270
assert len(val_examples) == 70

train_example_scenes = {
    x["scene_id"]
    for x in train_examples
}

val_example_scenes = {
    x["scene_id"]
    for x in val_examples
}

assert train_example_scenes.isdisjoint(
    val_example_scenes
)

print("=" * 112)
print("P16-C07 — DATASET GENERATED")
print("=" * 112)
print("Total examples      :", len(examples))
print("Train examples      :", len(train_examples))
print("Validation examples :", len(val_examples))
print("Train scenes        :", len(train_example_scenes))
print("Validation scenes   :", len(val_example_scenes))
print("Scene leakage       :", len(train_example_scenes & val_example_scenes))
print("P16-C07 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P16-C08 — GROUNDING / SAFETY VERIFIER
# ============================================================
packet_by_scene = {
    p["scene_id"]: p
    for p in packets
}

def resolve_path(obj, path):
    cur = obj

    for token in path.split("."):
        cur = cur[token]

    return cur

def canonical_fact_render(path, value):
    if value is None:
        return None

    if path.endswith(
        "fraction_of_coverage"
    ) or path.endswith(
        "review10_fraction"
    ) or path.endswith(
        "review20_fraction"
    ) or path.endswith(
        "low_pclear_fraction"
    ) or path.endswith(
        "sar_share"
    ) or path.endswith(
        "optical_share"
    ) or path.endswith(
        "flip_rate_no_sar"
    ) or path.endswith(
        "flip_rate_no_optical"
    ) or path.endswith(
        "high_fragility_tile_fraction"
    ):
        return pct(value)

    if path.endswith(
        "area_km2"
    ):
        return area(value)

    if path.endswith(
        "mean_entropy"
    ) or path.endswith(
        "mean_pclear"
    ) or path.endswith(
        "temporary_flood_js_sar"
    ) or path.endswith(
        "temporary_flood_js_optical"
    ):
        return dec(value)

    if path.endswith(
        "connected_components_8conn"
    ) or path.endswith(
        "largest_temporary_component_pixels"
    ):
        return integer(value)

    if path.endswith(
        "dominance"
    ):
        return str(value)

    return str(value)

verification_rows = []

for ex in examples:
    packet = packet_by_scene[ex["scene_id"]]

    errors = []

    # Source lineage.
    if ex["source_packet_sha256"] != packet["provenance"]["packet_sha256"]:
        errors.append("source packet SHA mismatch")

    if ex["source_segmenter"] != "FloodShift-04H-CAL":
        errors.append("wrong segmenter")

    if ex["source_test_status"] != "LOCKED_NOT_EVALUATED":
        errors.append("test status changed")

    # Factual-answer grounding.
    if ex["response_mode"] == "answer":
        for path in ex["grounded_fact_paths"]:
            value = resolve_path(packet, path)
            rendered = canonical_fact_render(path, value)

            # Optional facts may be absent and therefore intentionally omitted.
            if rendered is not None and rendered not in ex["response"]:
                errors.append(
                    f"rendered fact not present: {path} -> {rendered}"
                )

    # Refusal examples must not invent any scene-specific number.
    if ex["response_mode"] == "refuse":
        if re.search(r"\d+\.\d+%|\d+\.\d+\s*km²", ex["response"]):
            errors.append("refusal response contains unsupported numeric claim")

    # Never allow unsupported certainty language.
    banned_response_phrases = [
        "definitely caused by",
        "proves that",
        "ground truth confirms",
        "will definitely",
        "people were affected",
        "buildings were damaged",
    ]

    response_lower = ex["response"].lower()

    for phrase in banned_response_phrases:
        if phrase in response_lower:
            errors.append(f"banned phrase: {phrase}")

    verification_rows.append({
        "id": ex["id"],
        "scene_id": ex["scene_id"],
        "task": ex["task"],
        "response_mode": ex["response_mode"],
        "errors": errors,
        "pass": len(errors) == 0,
    })

verification_df = pd.DataFrame(verification_rows)

failed = verification_df[
    ~verification_df["pass"]
]

if len(failed):
    display(failed)
    raise AssertionError(
        f"{len(failed)} generated examples failed grounding verification."
    )

print("=" * 112)
print("P16-C08 — GROUNDING VERIFIER")
print("=" * 112)
print("Verified examples :", len(verification_df))
print("Failures          :", len(failed))
print("Scene leakage     : 0")
print("Frozen geo test   : untouched")
print("P16-C08 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P16-C09 — COPY SELF-CONTAINED EVIDENCE IMAGES
# ============================================================
copied = 0

for sid in sorted(scene_ids):
    src = PANEL_ROOT_07 / f"{sid}_evidence.png"
    dst = IMAGE_ROOT / f"{sid}_evidence.png"

    shutil.copy2(
        src,
        dst,
    )

    copied += 1

assert copied == 34
assert len(list(IMAGE_ROOT.glob("*.png"))) == 34

print("=" * 112)
print("P16-C09 — IMAGE PACK")
print("=" * 112)
print("Copied evidence panels:", copied)
print("Destination           :", IMAGE_ROOT)
print("Ground truth panels   : 0")
print("P16-C09 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P16-C10 — DATASET STATISTICS / BALANCE AUDIT
# ============================================================
examples_df = pd.DataFrame([
    {
        "id": x["id"],
        "scene_id": x["scene_id"],
        "split": x["split"],
        "task": x["task"],
        "response_mode": x["response_mode"],
        "response_chars": len(x["response"]),
    }
    for x in examples
])

task_stats = (
    examples_df
    .groupby(
        ["split", "task", "response_mode"],
        as_index=False,
    )
    .agg(
        examples=("id", "count"),
        scenes=("scene_id", "nunique"),
        mean_response_chars=("response_chars", "mean"),
    )
)

mode_counts = (
    examples_df["response_mode"]
    .value_counts()
    .rename_axis("response_mode")
    .reset_index(name="count")
)

review_scene_count = sum(
    scene_requires_review(p)
    for p in packets
)

print("=" * 112)
print("P16-C10 — DATASET AUDIT")
print("=" * 112)
print("Examples:", len(examples_df))
print()
print("Response modes:")
display(mode_counts)

print("Task balance:")
display(task_stats)

print(
    "Scenes triggering fixed review policy:",
    review_scene_count,
    "/",
    len(packets),
)

print("P16-C10 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P16-C11 — HUMAN-READABLE SAMPLE EXAMPLES
# ============================================================
sample_ids = []

for task in TASKS:
    candidate = next(
        x
        for x in examples
        if x["task"] == task
    )
    sample_ids.append(candidate["id"])

sample_examples = [
    x
    for x in examples
    if x["id"] in sample_ids
]

for ex in sample_examples:
    print("=" * 112)
    print("ID   :", ex["id"])
    print("TASK :", ex["task"])
    print("MODE :", ex["response_mode"])
    print("IMAGE:", ex["image"])
    print()
    print("USER:")
    print(ex["instruction"])
    print()
    print("ASSISTANT:")
    print(ex["response"])
    print()

print("P16-C11 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P16-C12 — EXPORT CANONICAL + LLAVA-STYLE DATASETS
# ============================================================
TRAIN_JSONL = DATASET_ROOT / "FloodShift_08_train.jsonl"
VAL_JSONL = DATASET_ROOT / "FloodShift_08_validation.jsonl"
ALL_JSONL = DATASET_ROOT / "FloodShift_08_all.jsonl"

LLAVA_TRAIN_JSON = DATASET_ROOT / "FloodShift_08_llava_train.json"
LLAVA_VAL_JSON = DATASET_ROOT / "FloodShift_08_llava_validation.json"

TASK_STATS_CSV = DATASET_ROOT / "FloodShift_08_task_stats.csv"
SPLIT_CSV = DATASET_ROOT / "FloodShift_08_scene_split.csv"
VERIFICATION_CSV = DATASET_ROOT / "FloodShift_08_grounding_verification.csv"

DATASET_CARD = DATASET_ROOT / "FloodShift_08_dataset_card.json"
PROTOCOL_PATH = OUTPUT_ROOT / "FloodShift_08_protocol_record.json"

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False,
                    sort_keys=True,
                    allow_nan=False,
                )
                + "\n"
            )

write_jsonl(
    TRAIN_JSONL,
    train_examples,
)

write_jsonl(
    VAL_JSONL,
    val_examples,
)

write_jsonl(
    ALL_JSONL,
    examples,
)

def to_llava(ex):
    return {
        "id": ex["id"],
        "image": ex["image"],
        "conversations": [
            {
                "from": "human",
                "value": (
                    "<image>\n"
                    + ex["instruction"]
                ),
            },
            {
                "from": "gpt",
                "value": ex["response"],
            },
        ],
        "metadata": {
            "scene_id": ex["scene_id"],
            "split": ex["split"],
            "task": ex["task"],
            "response_mode": ex["response_mode"],
            "grounded_fact_paths": ex["grounded_fact_paths"],
            "source_packet_sha256": ex["source_packet_sha256"],
        },
    }

with open(LLAVA_TRAIN_JSON, "w", encoding="utf-8") as f:
    json.dump(
        [to_llava(x) for x in train_examples],
        f,
        ensure_ascii=False,
        indent=2,
    )

with open(LLAVA_VAL_JSON, "w", encoding="utf-8") as f:
    json.dump(
        [to_llava(x) for x in val_examples],
        f,
        ensure_ascii=False,
        indent=2,
    )

task_stats.to_csv(
    TASK_STATS_CSV,
    index=False,
)

split_df[
    ["scene_id"]
].assign(
    split=lambda d: d["scene_id"].map(scene_split)
).to_csv(
    SPLIT_CSV,
    index=False,
)

verification_df.to_csv(
    VERIFICATION_CSV,
    index=False,
)

dataset_card = {
    "name": "FloodShift-08-Grounded-Multimodal-Instructions",
    "version": "1.0",
    "source": "FloodShift-07 deterministic evidence packets",
    "source_archive_sha256": ARCHIVE_07_SHA,
    "source_segmenter": "FloodShift-04H-CAL",
    "source_ground_truth_in_packets": False,
    "examples": len(examples),
    "train_examples": len(train_examples),
    "validation_examples": len(val_examples),
    "train_scenes": len(TRAIN_SCENES),
    "validation_scenes": len(VAL_SCENES),
    "scene_overlap": 0,
    "examples_per_scene": len(TASKS),
    "task_families": TASKS,
    "response_modes": {
        k: int(v)
        for k, v in Counter(
            x["response_mode"]
            for x in examples
        ).items()
    },
    "review_policy": {
        "review10_fraction_threshold": SCENE_REVIEW_PIXEL_FRACTION,
        "high_fragility_tile_fraction_threshold": SCENE_FRAGILITY_TILE_FRACTION,
        "role": "instruction-response policy only; not a segmentation metric",
    },
    "grounding": {
        "answers_generated_by_external_llm": False,
        "deterministic_templates": True,
        "fact_path_traceability": True,
        "verification_failures": int((~verification_df["pass"]).sum()),
    },
    "frozen_test_status": "LOCKED_NOT_EVALUATED",
}

with open(DATASET_CARD, "w", encoding="utf-8") as f:
    json.dump(
        dataset_card,
        f,
        indent=2,
    )

protocol08 = {
    "experiment": "FloodShift-08-Grounded-Instruction-Dataset",
    "source_07_archive_sha256": ARCHIVE_07_SHA,
    "source_segmenter": "FloodShift-04H-CAL",
    "dataset_examples": len(examples),
    "train_examples": len(train_examples),
    "validation_examples": len(val_examples),
    "train_scenes": len(TRAIN_SCENES),
    "validation_scenes": len(VAL_SCENES),
    "scene_overlap": 0,
    "grounding_verification_failures": int((~verification_df["pass"]).sum()),
    "external_llm_generation_used": False,
    "ground_truth_in_training_packets": False,
    "test_status": "LOCKED_NOT_EVALUATED",
    "next_stage": "variational_multimodal_language_model",
}

with open(PROTOCOL_PATH, "w", encoding="utf-8") as f:
    json.dump(
        protocol08,
        f,
        indent=2,
    )

print("=" * 112)
print("P16-C12 — DATASET EXPORT")
print("=" * 112)
print("Canonical train :", TRAIN_JSONL)
print("Canonical val   :", VAL_JSONL)
print("LLaVA train     :", LLAVA_TRAIN_JSON)
print("LLaVA val       :", LLAVA_VAL_JSON)
print("Dataset card    :", DATASET_CARD)
print("Protocol record :", PROTOCOL_PATH)
print("Images          :", len(list(IMAGE_ROOT.glob('*.png'))))
print("P16-C12 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P16-C13 — FINAL SELF-CONTAINED ARCHIVE + SHA256
# ============================================================
ARCHIVE_PATH = OUTPUT_ROOT / "FloodShift_08_ARCHIVE.zip"

with zipfile.ZipFile(
    ARCHIVE_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:

    for p in sorted(DATASET_ROOT.rglob("*")):
        if p.is_file():
            zf.write(
                p,
                arcname=p.relative_to(OUTPUT_ROOT),
            )

    zf.write(
        PROTOCOL_PATH,
        arcname=PROTOCOL_PATH.relative_to(OUTPUT_ROOT),
    )

ARCHIVE_SHA = sha256_file(
    ARCHIVE_PATH
)

print("=" * 112)
print("P16-C13 — FINAL EXPORT")
print("=" * 112)
print("Examples            :", len(examples))
print("Train / validation  :", len(train_examples), "/", len(val_examples))
print("Images              :", len(list(IMAGE_ROOT.glob('*.png'))))
print("Grounding failures  :", int((~verification_df["pass"]).sum()))
print("Scene leakage       :", len(TRAIN_SCENES & VAL_SCENES))
print("Ground truth packets: False")
print("Frozen geo test     : LOCKED_NOT_EVALUATED")
print("Archive             :", ARCHIVE_PATH)
print("Archive SHA256      :", ARCHIVE_SHA)
print()
print("NEXT: Gaussian-latent variational multimodal language model.")
print("P16-C13 COMPLETE ✅")


## After the run

Send me the executed notebook or these outputs:

- **P16-C07** — total/train/validation example counts
- **P16-C08** — grounding verifier result
- **P16-C10** — task/mode balance
- **P16-C11** — sample examples
- **P16-C12** — exported dataset paths
- **P16-C13** — archive SHA256

Download:

```text
FloodShift_08_ARCHIVE.zip
```

The archive is self-contained and includes:

```text
dataset/images/*.png

dataset/FloodShift_08_train.jsonl
dataset/FloodShift_08_validation.jsonl
dataset/FloodShift_08_all.jsonl

dataset/FloodShift_08_llava_train.json
dataset/FloodShift_08_llava_validation.json

dataset/FloodShift_08_dataset_card.json
dataset/FloodShift_08_task_stats.csv
dataset/FloodShift_08_scene_split.csv
dataset/FloodShift_08_grounding_verification.csv

FloodShift_08_protocol_record.json
```

This is the input package for the next stage: the **Gaussian-latent variational multimodal language model**.
